In [51]:
# Standard libraries
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
from typing import List, Tuple, Optional, Set, Dict
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Reproducibility
#RANDOM_SEED = 42
#np.random.seed(RANDOM_SEED)

# RDKitran
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, Draw
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit import DataStructs

# Scikit-learn
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
from scipy.stats import spearmanr

print("Core imports successful!")
print(f"RDKit version: {Chem.rdBase.rdkitVersion}")


# ChemProp import (may take a few seconds)
try:
    import chemprop
    print(f"ChemProp version: {chemprop.__version__}")
    CHEMPROP_AVAILABLE = True
except ImportError:
    print("ChemProp not installed. Install with: pip install chemprop")
    CHEMPROP_AVAILABLE = False

Core imports successful!
RDKit version: 2023.09.6
ChemProp not installed. Install with: pip install chemprop


In [55]:
# RF + Morgan fingerprintings

from random import random


def compute_morgan_fingerprint(mol: Chem.Mol, radius: int = 2, n_bits: int = 2048) -> np.ndarray:
    """Compute Morgan (ECFP) fingerprint.
    
    Args:
        mol: RDKit Mol object
        radius: Fingerprint radius (2 == ECFP4)
        n_bits: Number of bits (aka the 'length' of the fingerprint)
        
    Returns:
        numpy array of shape (n_bits,)
    """
    fingerpint = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    fingerpint = np.array(fingerpint)
    return fingerpint

def train_random_forest(
    X_train,y_train,X_val,y_val,X_test,y_test,
    n_estimators: int = 100,
    random_state: int = None ) -> Dict:
    if random_state is None:
        random_state = np.random.randint(0, 10000)  # Actually call the function
    
    print(random_state)
    model = RandomForestRegressor(n_estimators=n_estimators, random_state=random_state) #defining the model with the wanted hyperparameters
    model.fit(X_train, y_train) #fitting the model to the hyperparameters and the training data
    
    # Predictions
    pred_train = model.predict(X_train)
    pred_val = model.predict(X_val)
    pred_test = model.predict(X_test)
    
    # Metrics
    train_rmse = np.sqrt(mean_squared_error(y_train, pred_train))
    val_rmse = np.sqrt(mean_squared_error(y_val, pred_val))
    test_rmse = np.sqrt(mean_squared_error(y_test, pred_test))
    test_r2 = r2_score(y_test, pred_test)
    test_rho, _ = spearmanr(y_test, pred_test)
    test_mae = mean_absolute_error(y_test, pred_test)
    
    return {
        'model': model,
        'train_rmse': train_rmse,
        'val_rmse': val_rmse,
        'test_rmse': test_rmse,
        'test_r2': test_r2,
        'test_rho': test_rho,
        'test_predictions': pred_test,
        'test_measured': y_test,
        'test_mae': test_mae
    }

Now same process on the other datasets :

In [56]:
def run_randomforest (dataset,target): 
    test = pd.read_csv(f'../data/{dataset}_splits/{dataset}_test.csv')
    train = pd.read_csv(f'../data/{dataset}_splits/{dataset}_train.csv')
    val = pd.read_csv(f'../data/{dataset}_splits/{dataset}_val.csv')

    test['mol2'] = test['smiles'].apply(Chem.MolFromSmiles)
    train['mol2'] = train['smiles'].apply(Chem.MolFromSmiles)
    val['mol2'] = val['smiles'].apply(Chem.MolFromSmiles)

    test['mol'] = test['mol2']
    train['mol'] = train['mol2']
    val['mol'] = val['mol2']

    X_train = np.array([compute_morgan_fingerprint(mol) for mol in train['mol']])
    y_train = train[target].values
    X_val = np.array([compute_morgan_fingerprint(mol) for mol in val['mol']])
    y_val = val[target].values
    X_test = np.array([compute_morgan_fingerprint(mol) for mol in test['mol']])
    y_test = test[target].values  


    rf_morgan_scaffold = train_random_forest(X_train,y_train,X_val,y_val,X_test,y_test)
    print(f"  Test RMSE: {rf_morgan_scaffold['test_rmse']:.3f}, R²: {rf_morgan_scaffold['test_r2']:.3f}, ρ: {rf_morgan_scaffold['test_rho']:.3f}, MAE: {rf_morgan_scaffold['test_mae']:.3f}")
    return rf_morgan_scaffold 

In [57]:
for dataset_duo in [['lipo', 'exp'], ['qm7', 'u0_atom'], ['dr', 'DR']]:
    dataset = dataset_duo[0]
    target = dataset_duo[1] 
    print(f"Running Random Forest on {dataset} with target {target}...")
    run_randomforest(dataset,target)

Running Random Forest on lipo with target exp...
2483
  Test RMSE: 0.862, R²: 0.442, ρ: 0.665, MAE: 0.669
Running Random Forest on qm7 with target u0_atom...


[11:22:25] WARNING: not removing hydrogen atom without neighbors
[11:22:25] WARNING: not removing hydrogen atom without neighbors


4207
  Test RMSE: 192.734, R²: 0.236, ρ: 0.555, MAE: 141.039
Running Random Forest on dr with target DR...
9506
  Test RMSE: 0.474, R²: 0.224, ρ: 0.470, MAE: 0.368


In [59]:
for _ in range (5) :
    for dataset_duo in [['lipo', 'exp'], ['qm7', 'u0_atom'], ['dr', 'DR']]:
        dataset = dataset_duo[0]
        target = dataset_duo[1] 
        print(f"Running Random Forest on {dataset} with target {target}...")
        run_randomforest(dataset,target)

Running Random Forest on lipo with target exp...
715
  Test RMSE: 0.876, R²: 0.423, ρ: 0.647, MAE: 0.680
Running Random Forest on qm7 with target u0_atom...


[11:30:26] WARNING: not removing hydrogen atom without neighbors
[11:30:26] WARNING: not removing hydrogen atom without neighbors


7952
  Test RMSE: 193.248, R²: 0.232, ρ: 0.552, MAE: 141.085
Running Random Forest on dr with target DR...
5632
  Test RMSE: 0.457, R²: 0.278, ρ: 0.512, MAE: 0.355
Running Random Forest on lipo with target exp...
6788
  Test RMSE: 0.874, R²: 0.425, ρ: 0.642, MAE: 0.682
Running Random Forest on qm7 with target u0_atom...


[11:33:51] WARNING: not removing hydrogen atom without neighbors
[11:33:51] WARNING: not removing hydrogen atom without neighbors


2443
  Test RMSE: 193.260, R²: 0.232, ρ: 0.555, MAE: 140.631
Running Random Forest on dr with target DR...
7860
  Test RMSE: 0.463, R²: 0.261, ρ: 0.485, MAE: 0.357
Running Random Forest on lipo with target exp...
7531
  Test RMSE: 0.868, R²: 0.433, ρ: 0.655, MAE: 0.677
Running Random Forest on qm7 with target u0_atom...


...

7471
  Test RMSE: 191.002, R²: 0.250, ρ: 0.568, MAE: 139.194
Running Random Forest on dr with target DR...
7588
  Test RMSE: 0.466, R²: 0.252, ρ: 0.481, MAE: 0.355
Running Random Forest on lipo with target exp...
7863
  Test RMSE: 0.876, R²: 0.424, ρ: 0.641, MAE: 0.691
Running Random Forest on qm7 with target u0_atom...


[11:40:49] WARNING: not removing hydrogen atom without neighbors
[11:40:49] WARNING: not removing hydrogen atom without neighbors


390
  Test RMSE: 193.892, R²: 0.227, ρ: 0.549, MAE: 142.041
Running Random Forest on dr with target DR...
8764
  Test RMSE: 0.471, R²: 0.234, ρ: 0.486, MAE: 0.366
Running Random Forest on lipo with target exp...
3527
  Test RMSE: 0.869, R²: 0.432, ρ: 0.644, MAE: 0.679
Running Random Forest on qm7 with target u0_atom...


[11:44:19] WARNING: not removing hydrogen atom without neighbors
[11:44:19] WARNING: not removing hydrogen atom without neighbors


4411
  Test RMSE: 192.496, R²: 0.238, ρ: 0.557, MAE: 141.161
Running Random Forest on dr with target DR...
2651
  Test RMSE: 0.474, R²: 0.223, ρ: 0.445, MAE: 0.367
